# Урок 03 - Агентские шаблоны проектирования

В этом уроке мы рассмотрим три основных шаблона проектирования для создания эффективных ИИ-агентов:

1. **Четкие инструкции для агента** — создание точных, определяющих роль подсказок, которые направляют поведение агента
2. **Структурированный вывод с моделями Pydantic** — обеспечение возврата агентами предсказуемых и валидированных данных
3. **Агенты с одной ответственностью** — проектирование сфокусированных агентов, каждый из которых хорошо выполняет одну задачу

Мы применим каждый шаблон к сценарию **рекомендации туристических направлений**, постепенно создавая систему, которая сможет предлагать направления, проверять доступность и управлять логистикой.


## Настройка


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Шаблон 1: Четкие инструкции для агента

Самый эффективный шаблон также является самым простым: написание четких, подробных инструкций для вашего агента.

Хорошие инструкции определяют:
- **Кто** такой агент (персона и тон)
- **Что** он должен делать (пошаговые обязанности)
- **Как** он должен вести себя (ограничения и стиль)

Ниже мы создаем туристического консьержа с явными инструкциями, которые формируют каждый его ответ.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Шаблон 2: Структурированный вывод с моделями Pydantic

Свободный текст полезен для разговора, но последующие системы нуждаются в структурированных данных.
Сочетая **модели Pydantic** с **функцией инструмента**, мы можем:

- Определить точную схему для вывода агента
- Автоматически проверять ответы
- Надёжно интегрировать результаты агента в логику приложения

Ключ к обеспечению этого — передача `response_format` при запуске агента. Это заставляет
модель возвращать проверенный объект `TravelRecommendations` (доступен в `response.value`)
вместо свободного текста. Функция `get_destination_details` также возвращает типизированный
`DestinationRecommendation`, благодаря чему данные остаются структурированными от начала до конца.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## Шаблон 3: Агенты с одной ответственностью

Сложные задачи выигрывают от распределения работы между несколькими специализированными агентами, каждый из которых имеет одну ответственность:

- **Эксперт по направлению**, знающий о местах и доступности
- **Планировщик логистики**, который занимается рейсами, отелями и маршрутами

Это отражает принцип инженерии программного обеспечения *разделения ответственности* — каждый агент легче тестируется, поддерживается и улучшается независимо.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Резюме

В этом уроке мы применили три агентных шаблона проектирования к сценарию рекомендателя путешествий:

| Шаблон | Основная идея | Преимущество |
|---|---|---|
| **Четкие инструкции** | Определить персону, обязанности и ограничения заранее | Последовательное, соответствующее бренду поведение агента |
| **Структурированный вывод** | Использовать модели Pydantic как формат ответа | Проверенные, машиночитаемые результаты |
| **Единая ответственность** | Назначить каждому агенту одну конкретную задачу | Проще тестировать, поддерживать и комбинировать |

Эти шаблоны органично сочетаются — вы можете объединить четкие инструкции со структурированным выводом внутри агента с единой ответственностью, чтобы создавать надежные, готовые к производству системы.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:
Этот документ был переведен с использованием сервиса машинного перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия по обеспечению точности, имейте в виду, что автоматический перевод может содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для получения критически важной информации рекомендуется обратиться к профессиональному человеческому переводу. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования этого перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
